In [1]:
# 01_data_preprocessing.ipynb

import pandas as pd
import numpy as np
from pathlib import Path

# nicer display
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)
pd.options.display.float_format = "{:,.3f}".format

In [ ]:
# Define project paths (adapt if your structure is different)
DATA_DIR = Path("../data")   #creates 'Path' object pointing to a 'data' folder 1 level above notebook/script folder. (relative to where this code is located)     
RAW_PATH = DATA_DIR / "raw" / "ev_charging_patterns.csv"    # builds the 'Path' to where original/raw csv should live
PROCESSED_DIR = DATA_DIR / "processed"  # defines the folder for processed/cleaned outputs
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)    # Created the folder if it doesn't exist.
CLEAN_PATH = PROCESSED_DIR / "ev_charging_clean.csv"   # Builds the output path: ../data/processed/ev_charging_clean.csv → where you’ll save your cleaned dataset.

RAW_PATH, CLEAN_PATH  #In a Jupyter notebook, this just prints/returns both paths so you can quickly verify them.

(WindowsPath('../data/raw/ev_charging_patterns.csv'),
 WindowsPath('../data/processed/ev_charging_clean.csv'))

In [ ]:
# Reads the file located at 'RAW_PATH'
# Stores it in pandas DataFrame called 'df_raw'
df_raw = pd.read_csv(RAW_PATH)

print("Raw shape:", df_raw.shape)       # df_raw.shape returns (rows, columns)
df_raw.head()               # Shows the first 5 rows

Raw shape: (1320, 20)


,User ID,Vehicle Model,Battery Capacity (kWh),Charging Station ID,Charging Station Location,Charging Start Time,Charging End Time,Energy Consumed (kWh),Charging Duration (hours),Charging Rate (kW),Charging Cost (USD),Time of Day,Day of Week,State of Charge (Start %),State of Charge (End %),Distance Driven (since last charge) (km),Temperature (°C),Vehicle Age (years),Charger Type,User Type
0,User_1,BMW i3,108.463,Station_391,Houston,2024-01-01 00:00:00,2024-01-01 00:39:00,60.712,0.591,36.389,13.088,Evening,Tuesday,29.372,86.120,293.602,27.948,2.000,DC Fast Charger,Commuter
1,User_2,Hyundai Kona,100.000,Station_428,San Francisco,2024-01-01 01:00:00,2024-01-01 03:01:00,12.339,3.134,30.678,21.128,Morning,Monday,10.116,84.664,112.113,14.311,3.000,Level 1,Casual Driver
2,User_3,Chevy Bolt,75.000,Station_181,San Francisco,2024-01-01 02:00:00,2024-01-01 04:48:00,19.129,2.453,27.514,35.667,Morning,Thursday,6.855,69.918,71.799,21.002,2.000,Level 2,Commuter
3,User_4,Hyundai Kona,50.000,Station_327,Houston,2024-01-01 03:00:00,2024-01-01 06:42:00,79.458,1.266,32.883,13.036,Evening,Saturday,83.120,99.624,199.578,38.316,1.000,Level 1,Long-Distance Traveler
4,User_5,Hyundai Kona,50.000,Station_108,Los Angeles,2024-01-01 04:00:00,2024-01-01 05:46:00,19.629,2.020,10.216,10.161,Morning,Saturday,54.259,63.744,203.662,-7.834,1.000,Level 1,Long-Distance Traveler


In [ ]:
# prints a "health report" of DataFrame, a metadata summary
# RangeIndex: how many rows you have (e.g., RangeIndex: 50000 entries, 0 to 49999)
# Columns: each column name + its non-null count
# Dtype: the detected data type of each column (int64, float64, object, datetime64[ns], etc.)
# memory usage: roughly how much RAM the DataFrame uses

df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1320 entries, 0 to 1319
Data columns (total 20 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   User ID                                   1320 non-null   object 
 1   Vehicle Model                             1320 non-null   object 
 2   Battery Capacity (kWh)                    1320 non-null   float64
 3   Charging Station ID                       1320 non-null   object 
 4   Charging Station Location                 1320 non-null   object 
 5   Charging Start Time                       1320 non-null   object 
 6   Charging End Time                         1320 non-null   object 
 7   Energy Consumed (kWh)                     1254 non-null   float64
 8   Charging Duration (hours)                 1320 non-null   float64
 9   Charging Rate (kW)                        1254 non-null   float64
 10  Charging Cost (USD)                 

# Cell 4 – Copy and convert time columns to datetime

In [5]:
df = df_raw.copy()

# Convert start/end time to proper datetimes
time_cols = ["Charging Start Time", "Charging End Time"]
for col in time_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# Sanity check: any unparsable timestamps?
df[time_cols].isna().sum()


Charging Start Time    0
Charging End Time      0
dtype: int64

# Cell 5 – Create session / charge / delay / utilisation

In [6]:
# 1) Total session duration: plug-in to plug-out (ΔT_session)
df["session_duration_h"] = (
    df["Charging End Time"] - df["Charging Start Time"]
).dt.total_seconds() / 3600

# 2) Active charging duration as given (ΔT_charge)
df["charge_duration_h"] = df["Charging Duration (hours)"]

# 3) Delay / idle time while plugged-in but not charging (ΔT_delay)
df["delay_duration_h"] = df["session_duration_h"] - df["charge_duration_h"]

# 4) Utilisation of the session time
df["utilisation_rate"] = df["charge_duration_h"] / df["session_duration_h"]

df[["session_duration_h", "charge_duration_h", "delay_duration_h", "utilisation_rate"]].describe()

,session_duration_h,charge_duration_h,delay_duration_h,utilisation_rate
count,"1,320.000","1,320.000","1,320.000","1,320.000"
mean,2.225,2.269,-0.044,1.348
std,1.008,1.061,1.455,1.122
min,0.500,0.095,-5.459,0.033
25%,1.379,1.398,-1.058,0.636
50%,2.183,2.258,-0.001,1.000
75%,3.083,3.113,0.957,1.692
max,3.983,7.635,3.746,8.918


# Cell 7 – Filter invalid & absurd rows

- drop rows where any of these is NA:
 - session_duration_h, charge_duration_h, delay_duration_h, utilisation_rate
- drop rows where session_duration_h < charge_duration_h
- drop rows where delay_duration_h > 24 h (absurd idle times)

In [ ]:
dur_cols = ["session_duration_h", "charge_duration_h", "delay_duration_h", "utilisation_rate"]

before = df.shape[0]   # df.shape[0] = number of rows

# no missing in any of these
mask_not_na = df[dur_cols].notna().all(axis=1)
# .notna() produces a True/False table (True = value exists)
# .all(axis=1) means: for each row, True only if all 4 columns ar non-missing

# positive durations
mask_positive = (df["session_duration_h"] > 0) & (df["charge_duration_h"] > 0)
# Keeps rows where both durations are > 0

# session cannot be shorter than charge time
mask_session_ge_charge = df["session_duration_h"] >= df["charge_duration_h"]

# optional: drop absurdly long delays
MAX_DELAY_H = 24
mask_delay_reasonable = df["delay_duration_h"] <= MAX_DELAY_H

mask = mask_not_na & mask_positive & mask_session_ge_charge & mask_delay_reasonable

# df[mask] selects only rows where mask is True 
# .copy() avoids "view vs copy" issues in pandas and makes df a clean independent DataFrame
df = df[mask].copy()
after = df.shape[0]

print(f"Filtered by duration rules: {before} → {after} rows")
df[dur_cols].describe()  # shows new distributions after cleaning 


Filtered by duration rules: 1320 → 660 rows


,session_duration_h,charge_duration_h,delay_duration_h,utilisation_rate
count,660.000,660.000,660.000,660.000
mean,2.795,1.675,1.120,0.614
std,0.853,0.825,0.837,0.237
min,0.650,0.095,0.000,0.033
25%,2.179,0.967,0.441,0.441
50%,2.917,1.588,0.958,0.635
75%,3.554,2.261,1.631,0.818
max,3.983,3.778,3.746,1.000


# Cell 8 - handle numeric NAs: Energy (drop) + Distance (median)

In [ ]:
# 1) Drop rows with missing Energy Consumed (target later)
before = df.shape[0]
df = df.dropna(subset=["Energy Consumed (kWh)"]).copy()
# dropna(subset=[...]) removes any row where that column is NaN
# .copy() makes the result a clean independent DataFrame (avoids pandas "view" issues)
after = df.shape[0]

print(f"Dropped rows with missing Energy Consumed (kWh): {before} → {after}")
print("Remaining NAs in Energy Consumed (kWh):", df["Energy Consumed (kWh)"].isna().sum())
# .isna() turns values into booleans
# missing (NaN /NaT) -> True, not missing -> False
# .sum() then counts the True values, because in Python:
# True behaves like 1, False behaves like 0

Dropped rows with missing Energy Consumed (kWh): 660 → 622
Remaining NAs in Energy Consumed (kWh): 0


In [ ]:
# 2) Fill distance with median
distance_col = "Distance Driven (since last charge) (km)"  # stores string inside the column name

if distance_col in df.columns:                  # check the column actually exists, prevent code from crashing if dataset takde the col name
    median_distance = df[distance_col].median()     # Median is used instead of mean because it's more robust to outliers
    print("Median distance (km):", median_distance)

    df[distance_col] = df[distance_col].fillna(median_distance)
    print("Remaining NAs in distance:", df[distance_col].isna().sum())
else:
    print(f"Column {distance_col!r} not found – check column names.")

Median distance (km): 154.34561667139718
Remaining NAs in distance: 0


# Cell 9 – Recompute clean charging rate

In [10]:
# Charging power during actual charging
df["charging_rate_calc_kw"] = df["Energy Consumed (kWh)"] / df["charge_duration_h"]

print(df["charging_rate_calc_kw"].describe())

# Drop original noisy/partially missing column if present
if "Charging Rate (kW)" in df.columns:
    df = df.drop(columns=["Charging Rate (kW)"])

count   622.000
mean     35.362
std      46.140
min       0.083
25%      13.604
50%      24.986
75%      43.476
max     784.135
Name: charging_rate_calc_kw, dtype: float64


# Cell 10 - Final sanity checks 

In [11]:
# NA overview
df.isna().sum().sort_values(ascending=False)

# Quick stats for key numeric fields
df[
    [
        "session_duration_h",
        "charge_duration_h",
        "delay_duration_h",
        "utilisation_rate",
        "Energy Consumed (kWh)",
        "charging_rate_calc_kw",
        "Distance Driven (since last charge) (km)",
    ]
].describe()


,session_duration_h,charge_duration_h,delay_duration_h,utilisation_rate,Energy Consumed (kWh),charging_rate_calc_kw,Distance Driven (since last charge) (km)
count,622.000,622.000,622.000,622.000,622.000,622.000,622.000
mean,2.796,1.679,1.117,0.614,41.816,35.362,154.968
std,0.860,0.827,0.830,0.235,22.412,46.140,84.158
min,0.650,0.095,0.000,0.033,0.121,0.083,1.900
25%,2.171,0.964,0.442,0.442,23.548,13.604,82.288
50%,2.933,1.585,0.955,0.633,41.260,24.986,154.346
75%,3.567,2.284,1.628,0.818,60.722,43.476,220.089
max,3.983,3.724,3.746,1.000,152.239,784.135,398.365


# Cell 11 - Save cleaned dataset

In [12]:
df.to_csv(CLEAN_PATH, index=False)
print("Saved cleaned data to:", CLEAN_PATH)
print("Cleaned shape:", df.shape)

Saved cleaned data to: ..\data\processed\ev_charging_clean.csv
Cleaned shape: (622, 24)
